# reparameterization-trick — ex2: gradient flow check — backward populates grads on BOTH mu and logsigma

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reparameterization-trick`. Running the final beacon cell reports progress against the `VAE: Reparameterization trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: Reparameterization trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reparameterization-trick`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reparameterization-trick"
DD_SUBTOPIC = "VAE: Reparameterization trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Reparameterization trick — gradient flow check

Ex1 implemented `z = mu + sigma * eps`. The deepening move asks: WHY
did we go through that algebra at all? Because the gradient needs to
flow back to BOTH `mu` and `logsigma`.

```python
# WITH reparam — gradient flows:
z = mu + (0.5 * logsigma).exp() * eps   # eps = N(0,1).sample()
z.sum().backward()                       # mu.grad and logsigma.grad both populated

# WITHOUT reparam — gradient DOES NOT flow:
z = t.distributions.Normal(mu, sigma).sample()   # .sample() is non-differentiable
z.sum().backward()                                # RuntimeError or zero grads
```

**Sampling is the source of randomness; reparam moves it OUT of the
computational graph.** `eps ~ N(0, 1)` is sampled with `.detach()`-like
semantics. The DETERMINISTIC transform `mu + exp(0.5*logsigma) * eps`
is the only thing in the autograd path — so gradients flow through `mu`
(linearly) and through `logsigma` (via the exp factor on eps).

**Verification recipe.**
1. Create `mu`, `logsigma` as leaf tensors with `requires_grad=True`.
2. Reparameterize.
3. `z.sum().backward()`.
4. Assert `mu.grad is not None` (it should be `ones_like(mu)`).
5. Assert `logsigma.grad is not None` (it should be `0.5 * eps * exp(0.5*logsigma)`).

### Exercise 2 — gradient flow check — backward populates grads on BOTH mu and logsigma

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze gradient flow through the reparameterization trick: given leaf tensors `mu` and `logsigma` with `requires_grad=True`, the function `ex2_reparam_check` reparameterizes a sample, calls `backward` on `z.sum()`, and verifies that BOTH `mu.grad` and `logsigma.grad` are populated with the analytically expected values.
> Keywords: reparameterization, autograd, gradient, backward
> ```

**KCs targeted:** `reparam-puts-stochasticity-outside-graph`, `backward-populates-grad-on-both-distribution-params`

Implement `ex2_reparam_check(mu, logsigma, eps)`.

Inputs:
- `mu:       (B, L)` — leaf tensor with `requires_grad=True`.
- `logsigma: (B, L)` — leaf tensor with `requires_grad=True`.
- `eps:      (B, L)` — pre-sampled noise from `N(0, 1)`. Already detached (no grad). The function does NOT sample its own noise — tests need determinism.

Steps inside the function:
1. Reparameterize: `z = mu + (0.5 * logsigma).exp() * eps`. Same as `mu + sigma * eps` where `sigma = exp(0.5 * logsigma)`.
2. Compute `loss = z.sum()`.
3. Call `loss.backward()`.
4. Return a dict:
   ```
   {
       'z':            z.detach(),         # (B, L)
       'mu_grad':      mu.grad.clone(),    # (B, L) — should be ones
       'logsigma_grad': logsigma.grad.clone(),  # (B, L) — should be 0.5 * eps * exp(0.5*logsigma)
   }
   ```

**Don't zero gradients before calling.** Tests pre-zero them. Don't wrap in `no_grad`. The whole point is to LET autograd build the graph.

In [ ]:
def ex2_reparam_check(mu: Tensor, logsigma: Tensor, eps: Tensor) -> dict:
    """Reparameterize, backward on z.sum(), return grads on mu and logsigma."""
    raise NotImplementedError()


def _test_ex2():
    B, L = 4, 3
    t.manual_seed(7)
    mu_val       = t.randn(B, L)
    logsigma_val = t.randn(B, L) * 0.3
    eps          = t.randn(B, L)  # caller-provided noise — already detached

    mu       = mu_val.clone().requires_grad_(True)
    logsigma = logsigma_val.clone().requires_grad_(True)

    out = ex2_reparam_check(mu, logsigma, eps)

    # === Keys + shapes ===
    assert set(out.keys()) == {'z', 'mu_grad', 'logsigma_grad'}
    assert out['z'].shape             == (B, L)
    assert out['mu_grad'].shape       == (B, L)
    assert out['logsigma_grad'].shape == (B, L)

    # === z = mu + exp(0.5*logsigma) * eps  (forward correctness) ===
    expected_z = mu_val + (0.5 * logsigma_val).exp() * eps
    assert t.allclose(out['z'], expected_z, atol=1e-6), 'reparam formula mismatch'

    # === Grad on mu must be EXACTLY ones (z = mu + ..., d(z.sum())/d(mu) = 1) ===
    expected_mu_grad = t.ones(B, L)
    assert t.allclose(out['mu_grad'], expected_mu_grad, atol=1e-6), (
        f'mu.grad must be ones; got max-diff = {(out["mu_grad"] - expected_mu_grad).abs().max():.2e}'
    )

    # === Grad on logsigma is 0.5 * eps * exp(0.5*logsigma) ===
    # z = mu + exp(0.5*logsigma) * eps
    # dz/dlogsigma = exp(0.5*logsigma) * eps * 0.5
    # d(z.sum())/dlogsigma_{b,l} = same per element
    expected_logsigma_grad = 0.5 * eps * (0.5 * logsigma_val).exp()
    assert t.allclose(out['logsigma_grad'], expected_logsigma_grad, atol=1e-6), (
        f'logsigma.grad must equal 0.5 * eps * exp(0.5*logsigma); '
        f'max-diff = {(out["logsigma_grad"] - expected_logsigma_grad).abs().max():.2e}'
    )

    # === Neither grad is None (the failure mode if reparam is bypassed) ===
    assert out['mu_grad']       is not None
    assert out['logsigma_grad'] is not None

    # === eps had no grad to start with and gets no grad after backward ===
    assert eps.grad is None, 'eps should have no grad — it is the noise source, not a parameter'

    # === When logsigma is very negative, sigma is small, so logsigma_grad ≈ 0 ===
    # Bound: |grad| = 0.5 * |eps| * exp(0.5*logsigma). With logsigma=-30 → sigma≈3.1e-7, grad ~ 1.5e-7 per element.
    mu2       = t.zeros(2, 2, requires_grad=True)
    logsigma2 = t.full((2, 2), -30.0, requires_grad=True)
    eps2      = t.randn(2, 2)
    out2 = ex2_reparam_check(mu2, logsigma2, eps2)
    assert out2['logsigma_grad'].abs().max() < 1e-5, (
        f'with logsigma=-30, sigma is tiny so logsigma.grad must be near zero; got {out2["logsigma_grad"].abs().max():.2e}'
    )
    # But mu.grad is still exactly 1.
    assert t.allclose(out2['mu_grad'], t.ones(2, 2), atol=1e-6)

    # === eps = 0 forces logsigma_grad to exactly 0 (since dz/dlogsigma scales with eps) ===
    mu3       = t.randn(3, 3, requires_grad=True)
    logsigma3 = t.randn(3, 3, requires_grad=True)
    eps3      = t.zeros(3, 3)
    out3 = ex2_reparam_check(mu3, logsigma3, eps3)
    assert t.allclose(out3['logsigma_grad'], t.zeros(3, 3), atol=1e-7), (
        'when eps=0, sigma cancels out → no gradient on logsigma'
    )
    # mu.grad still ones.
    assert t.allclose(out3['mu_grad'], t.ones(3, 3), atol=1e-6)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_reparam_check(mu, logsigma, eps):
    sigma = (0.5 * logsigma).exp()
    z = mu + sigma * eps
    loss = z.sum()
    loss.backward()
    return {
        'z':             z.detach(),
        'mu_grad':       mu.grad.clone(),
        'logsigma_grad': logsigma.grad.clone(),
    }
```

**Why mu.grad is exactly 1.** `z = mu + (sigma * eps)`. The `mu` branch enters linearly, so `dz/dmu = 1` element-wise. Backprop on `z.sum()` adds these element-wise grads — every entry of `mu.grad` is exactly 1.

**Why logsigma.grad scales with eps.** `dz/dlogsigma = (d/dlogsigma)[exp(0.5*logsigma) * eps] = 0.5 * exp(0.5*logsigma) * eps`. When `eps` is large the gradient through the variance branch is large; when `eps = 0` the gradient is exactly zero — the model can't learn `logsigma` from samples where the noise didn't kick.

**The non-reparameterized failure mode.** If you'd written `z = t.distributions.Normal(mu, sigma).sample()`, the `.sample()` call cuts the autograd graph — `z.requires_grad` would be False, and `backward` would either raise or silently leave both grads as None. The reparam trick exists precisely to keep the graph intact.

**`mu.grad.clone()` over `mu.grad`.** A subsequent backward call in the caller's code could accumulate INTO `mu.grad`. Returning a clone prevents downstream side-effects from mutating our returned values.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()